# Job Listing Prioritization for Career Changers: Profiles (career switches)

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Finds **real career changers** (anyone switching from their current background to a different role or field) in a public resume dataset: people who moved from another field into a **computer and mathematical** role. Each one gives the fit model (Option B, `docs/fit_analysis.md`) and role discovery a **pre-switch profile** (jobs and education before the switch) and a **real label** (the role they moved into).

**How a career changer is found.** Each resume is read as one timeline: jobs and education in date order, each entry labelled tech/math, other field, unclear, or skip (internships, high school). The **original background** is everything before the first tech or math entry. **Nobody who reached a computer and mathematical job is thrown away:** each person gets labels (origin, retrained, first tech job level, every tech role reached), and `status` names the main one. **Career changers are a view,** in three pools: **retrained after working in another field** and **retrained as a new grad from another field** (another field, then a tech or math program, then the tech job), and **changed roles within tech** (a move into a different tech role bucket after 12+ months in the first). Any job level counts, including manager. The ML model can learn from everyone, with origin as a feature; results are reported on career changers. The rules live in `src/career_switch.py`, versioned (`RULES_VERSION`) and logged in section 5.

**Data:** Kaggle `suriyaganesh/resume-dataset-structured` (MIT license, version 2 of 2024-11-14): about 54,000 resume records, converted by OCR and an LLM, so expect extraction errors. Stored in git-ignored `data/external/resume_structured/`.

**Data rules:** names and contact details (`people.csv`), employers and locations are never loaded. Aggregates only, except the cells tagged `data-rows` (the 4a.2d preview and the 4c review sample): clear it before committing. Never paste rows into AI tools.

**Leakage rule:** the pre-switch profile uses only jobs and education that started before the switch. Undated programs are placed by degree level (flagged in `study_undated`). Skills aren't dated, so they stay out of the pre-switch profile.

### Clarifying questions
- How many people switched into a computer and mathematical role by retraining, and from which backgrounds?
- Which fields did they train into, and which roles did that lead to? This is the role-discovery target list.
- How often do the rules mistake a title change, a student job or a technical background for a career change?

### Roadmap
1. Setup · 2. Load, then remove duplicate resumes (2b) · 3. Dates and title flags
4. Find the switches; viable candidates (4a.1); the retrained table (4a.2); changed roles within tech (4a.3); the review round (4b, 4c)
5. Rules log · 6. Freeze and save

## 1. Setup

In [ ]:
# 1a Libraries, shared helpers and the career-switch rules
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from scipy import stats
from sklearn.linear_model import LogisticRegression

from src.eda_helpers import BLUE, DATA_DIR, INK_2, MUTED, ORANGE, bar_h, finish, heatmap, show, use_chart_style
from src.career_switch import (REASONS, RULES, RULES_VERSION, background_area, find_switches, person_timeline,
                               prepare_education, review_summary, tag_jobs)
use_chart_style()

RESUME_DIR = DATA_DIR / "external" / "resume_structured"
REVIEW_DIR = DATA_DIR / "profiles" / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("rules version:", RULES_VERSION)

## 2. Load

Only the columns the rules need. If the dataset's headers differ from its documentation, this stops and lists what's missing.

In [ ]:
# 2a Find the files and load only the columns we need
import re
import zipfile

def is_named(path, name):
    """Match a file name, ignoring case and a leading number prefix (the zip has 04_experience.csv)."""
    return re.sub(r"^\d+_", "", Path(path).name.lower()) == name.lower()

def unzip_if_needed():
    """If the CSVs aren't unzipped yet, extract the Kaggle zip found anywhere under data/ (it must contain experience.csv)."""
    if any(is_named(p, "experience.csv") for p in RESUME_DIR.rglob("*")):
        return
    zips = sorted(DATA_DIR.rglob("*.zip"))
    for zp in zips:
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            if not any(is_named(n, "experience.csv") for n in names):
                continue
            if any(Path(n).is_absolute() or ".." in Path(n).parts for n in names):
                raise ValueError(f"{zp.name} has unsafe paths; not extracting")
            RESUME_DIR.mkdir(parents=True, exist_ok=True)
            z.extractall(RESUME_DIR)
        print(f"unzipped {zp.relative_to(DATA_DIR.parent)} into {RESUME_DIR.relative_to(DATA_DIR.parent)} ({len(names)} files); you can delete the zip")
        return
    # Nothing matched: report file names only (never contents), so the problem can be fixed
    print("No zip under data/ contains experience.csv. Zips found:", [str(z.relative_to(DATA_DIR.parent)) for z in zips] or "none")
    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as z:
                print(f"  {zp.name}: {len(z.namelist())} files; CSVs: {[Path(n).name for n in z.namelist() if n.lower().endswith('.csv')][:20]}")
        except zipfile.BadZipFile:
            print(f"  {zp.name}: not a valid zip (still downloading, or the download failed)")

def find_csv(name):
    unzip_if_needed()
    hits = sorted(p for p in RESUME_DIR.rglob("*") if is_named(p, name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {RESUME_DIR}. Download the dataset first (see docs/fit_analysis.md).")
    return hits[0]

def load(name, cols):
    path = find_csv(name)
    header = pd.read_csv(path, nrows=0).columns
    missing = [c for c in cols if c not in header]
    if missing:
        raise KeyError(f"{name}: missing columns {missing}; found {list(header)}")
    return pd.read_csv(path, usecols=cols, dtype=str, encoding_errors="replace")

experience = load("experience.csv", ["person_id", "title", "start_date", "end_date"])
education = load("education.csv", ["person_id", "program", "start_date"])

show("Loaded tables (shape, and the share of each column filled)", pd.DataFrame({
    name: {"rows": len(df), "people": df.person_id.nunique(),
           **{f"{c} filled": round(df[c].notna().mean(), 3) for c in df.columns if c != "person_id"}}
    for name, df in {"experience": experience, "education": education}.items()
}).T)

### 2b Duplicates

The dataset combines several resume collections, and the first run's counts were almost all multiples of 3, which suggests each resume appears about three times under different `person_id`s. Duplicates would show the same person twice in a review round and put copies of one person on both sides of a train/test split (leakage).

Each person gets a **fingerprint**: every job (title, start and end date as written) plus every program (name and start date), sorted. People with the same fingerprint are the same resume; the lowest `person_id` is kept. Only **exact** copies are caught: a copy with an OCR difference gets a different fingerprint.

In [ ]:
# 2b Fingerprint every resume and keep one copy of each
from src.career_switch import fingerprints

DEDUPE = True
fp = fingerprints(experience, education)
N_BEFORE_DEDUPE = len(fp)
if DEDUPE:
    order = np.argsort(pd.to_numeric(fp.index, errors="coerce"), kind="stable")
    keep = fp.iloc[order].drop_duplicates().index  # the lowest person_id of each fingerprint
    experience = experience[experience.person_id.isin(keep)]
    education = education[education.person_id.isin(keep)]

show("Duplicate resumes", pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "duplicates removed": N_BEFORE_DEDUPE - experience.person_id.nunique(),
    "resumes left": experience.person_id.nunique(),
}).to_frame("resumes"))

**What the data shows**
- **54,933 resume records held 17,952 unique people:** 36,981 were duplicates, so each resume appeared about 3 times.

## 3. Dates and title flags

Dates become months (`MM/YYYY`; a bare year becomes January; "Present" becomes the dataset's update month, 2024-11). Titles are matched against the rules: which computer and mathematical bucket they fall in, and whether they're senior, school or internship work, or already technical. 3b shows the *patterns* of dates the parser still can't read (# for a digit, Word for letters), so the parser can be extended without looking at any dates.


In [ ]:
# 3a Parse dates and flag titles
jobs = tag_jobs(experience)
show("Computer and mathematical jobs by bucket", jobs.cm_bucket.value_counts().to_frame("jobs"))

In [ ]:
# 3b Date formats the parser can't read yet: patterns only (# = a digit, Word = letters), never the dates
def date_pattern(value):
    """'Jun 2015' -> 'Word ####', '06/15' -> '##/##'."""
    return re.sub(r"[A-Za-z]+", lambda m: "Word" if m.group(0)[0].isupper() else "word", re.sub(r"\d", "#", value.strip()))

unread = {}
for raw_col, parsed_col, label in [("start_date", "start", "start dates"), ("end_date", "end", "end dates")]:
    filled = jobs[raw_col].notna() & jobs[raw_col].astype(str).str.strip().ne("")
    unread[label] = jobs.loc[filled & jobs[parsed_col].isna(), raw_col].astype(str).map(date_pattern).value_counts()
patterns = pd.DataFrame(unread).fillna(0).astype(int)
patterns["share of all unreadable"] = (patterns.sum(axis=1) / patterns.sum().sum()).round(3)
patterns = patterns.sort_values("share of all unreadable", ascending=False).head(15).rename_axis("date pattern")
show(f"Unreadable dates by pattern (# = a digit, Word = letters): {unread['start dates'].sum():,} start dates and "
     f"{unread['end dates'].sum():,} end dates still unreadable", patterns)

## 4. Find the switches

For each person, in date order: their **first computer and mathematical job** is the switch, and the non-computer-and-mathematical jobs before it are their **background**. Everything before it is read as one timeline with their education. Their **original background** is everything before their first tech or math entry (a tech job, or a tech or math program). A person is a **candidate** if that background includes another field **and** they trained into tech or math before the switch: `candidate: retrained, new grad from another field` or `candidate: retrained, after working in another field`. Each rejection is kept with its reason, so we can review the rejected pools too.

Rules version 6 labels instead of rejecting: every person who reached a computer and mathematical job stays in `switches`, with `origin`, `retrained`, `first_level` and `roles_reached` (every tech role they held, not just the first). 4a's second cell counts people by label.


In [ ]:
# 4a One row per person: candidate, or the first rule that rejected them
switches = find_switches(jobs, education)
funnel = switches.status.value_counts().rename_axis("status").to_frame("people")
funnel["share"] = (funnel.people / len(switches)).round(3)
show(f"People by status (rules version {RULES_VERSION})", funnel)

fig, ax = plt.subplots(figsize=(9, 4.2))
bar_h(ax, funnel.people, f"People by status, rules version {RULES_VERSION}", "people")
finish()

In [ ]:
# 4a Labels for everyone who reached a computer and mathematical job
reached = switches[switches.target_bucket.notna()]
show(f"Everyone who reached a computer and mathematical job ({len(reached):,} people): origin by retraining",
     pd.crosstab(reached.origin.fillna("none"), reached.retrained.map({True: "retrained", False: "no tech or math training"}),
                 margins=True, margins_name="total"))
show("Level of the job at the switch", reached.first_level.value_counts().rename_axis("first_level").to_frame("people"))
in_view = reached.status.str.startswith("candidate")
data_counts = pd.DataFrame({
    "switched into data and analytics": reached.target_bucket.eq("data and analytics"),  # within tech: the role moved into
    "reached data and analytics at any point": reached.reached_data,
})
show("Data and analytics: everyone vs. the career-changer view", pd.DataFrame({
    "everyone who reached tech": data_counts.sum(), "career changers (view)": data_counts[in_view].sum()}))

**What the data shows**
- **99.7% of people have a readable job date** (17,904 of 17,952), up from 63% before the date fix (rules version 7).
- **15,108 people (84%) held a computer and mathematical job;** 2,539 of them are career changers (4a.1).

### 4a.1 Viable candidates

How many people the rules leave, and who they are in numbers. **Viable** means they passed the rules, not that they've been reviewed, so the counts are upper bounds until the review rounds. Counts are of unique people (after 2b). Aggregates only.

In [ ]:
# 4a.1a How many: from every resume to viable candidates
POOLS = ["candidate: retrained, new grad from another field", "candidate: retrained, after working in another field",
         "candidate: changed roles within tech"]
viable = switches[switches.status.isin(POOLS)].copy()
viable["path"] = viable.status.str.removeprefix("candidate: ").str.removeprefix("retrained, ")
viable["background_area"] = np.where(viable.origin_type.eq("job"), viable.source_title, viable.source_study)
viable["background_area"] = viable.background_area.map(background_area)
viable.loc[viable.from_bucket.notna(), "background_area"] = "tech: " + viable.from_bucket.dropna()
viable["switch_year"] = viable.switch_month.map(lambda m: m.year if pd.notna(m) else np.nan)
viable["years_worked_before"] = viable.background_months / 12
viable["programs"] = education.groupby("person_id").size().reindex(viable.index, fill_value=0)

steps = pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "unique people": len(switches),
    "with a readable job date": (switches.status != "no dated jobs").sum(),
    "ever held a computer and mathematical job": switches.target_bucket.notna().sum(),
    "viable career changers (total)": len(viable),
    **{f"  {p}": (viable.path == p).sum() for p in viable.path.value_counts().index},
})
show(f"From every resume to viable candidates (rules version {RULES_VERSION})",
     pd.DataFrame({"people": steps, "share of unique people": (steps / len(switches)).round(3)}))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(steps.index[::-1].str.strip(), steps.values[::-1], color=[ORANGE if s.startswith("  ") else BLUE for s in steps.index[::-1]], height=0.65)
for y, v in enumerate(steps.values[::-1]):
    ax.text(v, y, f" {v:,}", va="center", color=INK_2, fontsize=8.5)
ax.set(title="From every resume to viable candidates", xlabel="people (orange: the paths)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
finish()

**What it shows**
- **2,539 career changers (14% of unique people):** 1,836 changed roles within tech, 364 retrained after working in another field and 339 retrained as new grads from another field.
- **The biggest drop is from the 15,108 who reached tech:** most didn't change field or role bucket in a way the pools count.

In [ ]:
# 4a.1b Who they are, in numbers: summary statistics, distributions and correlations
METRICS = {
    "switch_year": "switch year",
    "years_worked_before": "years worked before",
    "n_jobs_before": "jobs before the switch",
    "n_jobs": "jobs on the resume",
    "programs": "programs on the resume",
}
m = viable[list(METRICS)].rename(columns=METRICS).astype(float)
gap = viable.loc[viable.origin_type.eq("job"), "gap_months"].rename("gap before the switch, months (after-work path)")
both = pd.concat([m, gap], axis=1)
summary = both.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
summary["skew"] = both.skew()
show("Summary statistics, all viable candidates", summary.round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for ax, col, subset, note in [(axes[0], "switch year", m["switch year"], ""),
                              (axes[1], "years worked before", m["years worked before"][m["years worked before"] > 0], "\n(only people who worked before)")]:
    x = subset.dropna()
    counts, edges, _ = ax.hist(x, bins="auto", color=BLUE, alpha=0.85)
    grid = np.linspace(edges[0], edges[-1], 200)
    ax.plot(grid, stats.norm.pdf(grid, x.mean(), x.std()) * len(x) * np.diff(edges).mean(), color=ORANGE,
            label="bell curve with the same mean and std")
    ax.set(title=f"{col.capitalize()} (n = {len(x):,}, skew {x.skew():.1f}){note}", xlabel=col, ylabel="people")
    ax.legend(fontsize=8)
heatmap(axes[2], m.loc[:, m.std() > 0].corr(method="spearman"), "Spearman correlation", fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1)
finish()

**What it shows**
- *To fill in after running: typical (median) switch year and years worked before; how far each histogram is from the bell curve (skew above about 1 means a long right tail, so medians describe it better than means); and any correlation stronger than ±0.3.*

### 4a.2 Retrained into tech or math: what they trained into, and where it led

One row per viable candidate, stating the **field they trained into** (grouped by `TRAINING_FIELDS` in `src/career_switch.py`) and the program as written. The full table is saved to `data/profiles/` for browsing; everything printed here is aggregate except the preview in 4a.2d.

In [ ]:
# 4a.2a The retrained table, saved, with totals by path
RETRAINED_COLS = {
    "path": "path",
    "background_area": "original background (area)",
    "source_title": "main background job",
    "source_study": "field studied first",
    "retrain_field": "field trained into (grouped)",
    "retrain_study": "program trained in",
    "target_bucket": "first tech role (bucket)",
    "target_title": "first tech job title",
    "roles_reached": "tech roles reached",
    "switch_period": "when they switched",
    "background_months": "months worked in another field",
    "study_undated": "relied on an undated program",
}
FIELD, ROLE, AREA, WHEN = "field trained into (grouped)", "first tech role (bucket)", "original background (area)", "when they switched"
retrained = viable[viable.status.str.startswith("candidate: retrained")][list(RETRAINED_COLS)].rename(columns=RETRAINED_COLS).sort_values([FIELD, ROLE])
RETRAINED_PATH = DATA_DIR / "profiles" / f"retrained_candidates_v{RULES_VERSION}.csv"
retrained.to_csv(RETRAINED_PATH)
print(f"saved {len(retrained):,} rows to {RETRAINED_PATH.relative_to(DATA_DIR.parent)}")

by_path = retrained.path.value_counts()
show("Retrained career changers by path", pd.concat([by_path, pd.Series({"total": by_path.sum()})])
     .rename_axis("path").to_frame("people").assign(share=lambda d: (d.people / by_path.sum()).round(3)))

# One bar for the total, split by path
fig, ax = plt.subplots(figsize=(9, 1.9))
left = 0
for (path, n), color in zip(by_path.items(), [BLUE, ORANGE]):
    ax.barh([0], [n], left=left, color=color, height=0.6)
    ax.text(left + n / 2, 0, f"{path}\n{n:,} ({n / by_path.sum():.0%})", ha="center", va="center",
            color="white", fontsize=9, fontweight="bold")
    left += n
ax.set(title=f"Retrained career changers: {by_path.sum():,} in total", xlim=(0, by_path.sum()), yticks=[], xlabel="people")
ax.grid(False)
for side in ("left", "top", "right"):
    ax.spines[side].set_visible(False)
finish()

In [ ]:
# 4a.2b What they trained into, where they came from, and where it led
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
bar_h(axes[0], retrained[FIELD].value_counts(), "Field trained into", "people")
bar_h(axes[1], retrained[AREA].value_counts().head(10), "Original background (top 10 areas)", "people", color=MUTED)
bar_h(axes[2], retrained[ROLE].value_counts(), "First tech role", "people", color=ORANGE)
finish()

# Retraining compass, the past side: where each training field led (share of its people; cells under 5 people hidden)
led = pd.crosstab(retrained[FIELD], retrained[ROLE])
compass = led.div(led.sum(axis=1), axis=0).where(led >= 5).round(2)
compass["people"] = led.sum(axis=1)
show("Retraining compass: share of each training field's people by first tech role (blank = under 5 people)",
     compass.sort_values("people", ascending=False))

programs = retrained["program trained in"].dropna().str.lower().str.strip().value_counts()
show("Most common programs trained in, as written (top 10, shared by at least 5 people): checks the grouping",
     programs[programs >= 5].head(10).rename_axis("program").to_frame("people"))

**What it shows**
- *To fill in after running: the top fields trained into, the top backgrounds, the top first tech roles, and the strongest training → role links in the compass table (they feed the retraining compass with EDA_SDA_2's 2026 side). Flag any program in the top 10 that's in the wrong group.*

In [ ]:
# 4a.2c When: career-change dates by field, and whether the path changed over time
PERIODS = ["before 2000", "2000-2004", "2005-2009", "2010-2014", "2015-2019", "2020 or later", "not dated"]
when = pd.crosstab(retrained[FIELD], retrained[WHEN]).reindex(columns=PERIODS, fill_value=0)
when = when.loc[:, when.sum() > 0]  # drop empty periods
when = when.loc[when.sum(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(17, 0.42 * len(when) + 2.2), gridspec_kw={"width_ratios": [1.3, 1]})
heatmap(axes[0], when, "Field trained into × when they switched (people)")

d = viable[["switch_year"]].assign(new_grad=viable.path.eq("new grad from another field").astype(int)).dropna()
yearly = d.groupby("switch_year").new_grad.agg(["mean", "size"])
axes[1].scatter(yearly.index, yearly["mean"], s=15 + 185 * yearly["size"] / yearly["size"].max(),
                color=BLUE, alpha=0.7, label="share each year (dot size = people)")
if d.new_grad.nunique() == 2:
    logit = LogisticRegression().fit(d[["switch_year"]] - 2010, d.new_grad)
    years = np.arange(d.switch_year.min(), d.switch_year.max() + 1)
    axes[1].plot(years, logit.predict_proba(pd.DataFrame({"switch_year": years - 2010}))[:, 1], color=ORANGE,
                 label=f"logistic fit: odds × {np.exp(5 * logit.coef_[0][0]):.2f} every 5 years")
axes[1].set(title="Share who retrained as new grads (vs. after working), by switch year",
            xlabel="year of the switch", ylabel="share new grads", ylim=(-0.02, 1.02))
axes[1].legend(fontsize=8)
finish()

**What it shows**
- *To fill in after running: which periods most switches come from, whether some fields are recent (for example data science), and whether retraining shifted toward new grads or toward people who'd already worked (odds above 1 every 5 years means more new grads over time). Switch date is descriptive only; it never filters anyone.*

In [ ]:
# 4a.2d Preview: the first rows of the retrained table (shows rows: clear before committing)
display(retrained.head(20))

### 4a.3 Changed roles within tech

People already in tech who moved into a **different role bucket** after at least 12 months in their first one (for example IT support → data and analytics). For a user who already works in tech, these are the comparable paths. Aggregates only.

In [ ]:
# 4a.3a From which tech role to which (share of each starting role's people; cells under 5 people hidden)
within = viable[viable.status == "candidate: changed roles within tech"]
if within.empty:
    print("No one changed roles within tech under the current rules.")
else:
    moves = pd.crosstab(within.from_bucket, within.target_bucket)
    table = moves.div(moves.sum(axis=1), axis=0).where(moves >= 5).round(2)
    table["people"] = moves.sum(axis=1)
    show(f"Changed roles within tech ({len(within):,} people): starting role by the role they moved into (blank = under 5 people)",
         table.sort_values("people", ascending=False))
    show("Role they moved into", within.target_bucket.value_counts().rename_axis("role").to_frame("people"))

**What it shows**
- **1,836 people changed roles within tech.** IT support staff mostly moved into networks and systems (54%) or security (31%); developers into databases (31%), data and analytics (30%) or networks (25%); DBAs into software (33%), networks (28%) or data and analytics (26%).
- **About 210 moved into data and analytics,** mostly from software development (about 135) and databases (about 55). It's the smallest destination, so the compass shows it with its real share and the model weights it in training.

### 4b Review round

Set `POOL` (one of the two retrained pools, or a rejection status from 4a to check whether a rule throws real switches away), then run 4c. Each run shows **5 random people** from that pool, new ones every time, with each person's **whole timeline** (titles and programs with dates; no employers or locations), every entry's label and the switch marked. The printed seed brings a sample back (`SEED = <number>`). Tell Claude which `person_id`s look right or wrong, and why, in your own words; don't paste the timelines.

In [ ]:
# 4b Review settings
POOL = "candidate: retrained, new grad from another field"
# or "candidate: retrained, after working in another field", or "candidate: changed roles within tech"
N = 5
show("Reason codes", pd.Series(REASONS, name="meaning").to_frame())

In [ ]:
# 4c Five random people from POOL, new ones every run (shows rows: clear before committing)
SEED = int(np.random.default_rng().integers(1_000_000))  # replace with a printed number to see that sample again
pool = switches[switches.status == POOL]
sample = pool.sample(min(N, len(pool)), random_state=SEED)
print(f"{POOL}: {len(pool):,} people | seed {SEED}: rerun for new people, or set SEED = {SEED} to see these again")

edu_by = prepare_education(education)
for pid, s in sample.iterrows():
    tl = person_timeline(jobs[jobs.person_id == pid], edu_by.get(pid), s.switch_month)
    tl["switch"] = np.where(tl.target_job & tl.start.eq(s.switch_month), "<- switch", "")
    print(f"\nperson {pid} | first tech role: {s.target_bucket} | all tech roles: {s.roles_reached} | trained into: {s.retrain_field}")
    display(tl[["kind", "text", "start", "end", "label", "dated", "switch"]])

## 5. Rules log

One row per rules version: what changed and why, and the review result that prompted it.

| Version | Change | Why | Candidate precision (round) |
|:--|:--|:--|:--|
| 1 | First rules: first computer and mathematical job = the switch; reject senior first roles, school-only backgrounds, technical backgrounds, and backgrounds under 12 months | Strict start: precision first, then check the rejected pools for real switches | *not reviewed* |
| 2 | **New grads:** people with no real job before the switch are judged on what they studied. Candidate if they studied a field other than tech or math; rejected if they studied tech or math, only high school, or the field is unclear. Job switchers who studied tech are flagged (`studied_tech`), not rejected. Candidate statuses split into `candidate: changed jobs` and `candidate: new grad from another field` | Jacob: version 1 counted many new grads; a new grad only counts if they studied a different background, not math or tech | *not reviewed* |
| 3 | **Whole-resume timeline:** jobs and education merged in date order and labelled; the original background is everything before the first tech or math entry. Career changer if it includes another field. New pool `candidate: retrained into tech or math` (another field, then a tech or math program, then the tech job). Undated programs placed by degree level. Technical-sounding job titles count as tech entries instead of rejecting the person outright | Jacob: people may study something unrelated first, then study again into math or tech; read resumes holistically | *not reviewed* |
| 4 | **Only retrained people are candidates**, in two pools: `candidate: retrained, new grad from another field` and `candidate: retrained, after working in another field`. Direct switches with no tech or math training get non-candidate statuses (`keep_direct_switches` = False). Also: duplicate resumes removed before the rules run (2b) | Jacob: we only care about career changers who retrained into tech or math | *not reviewed*; 354 candidates |
| 5 | **Tech titles:** security analyst/engineer/specialist, network, PC technician/support, webmaster and database titles count as tech. **Vague degrees:** "technology" or "engineering" alone names no field, so "Bachelor of Technology in Technology" is unclear | The version 4 run: these titles appeared as other-field backgrounds, and vague degrees counted as another field | *not reviewed* |
| 6 | **Label instead of reject:** everyone who reached a computer and mathematical job is kept, with `origin`, `retrained`, `first_level` and `roles_reached` (every tech role held); career changers are a view. Only a **manager-level** first tech job is left out of the view (senior is kept, matching the listings side). **Wider data titles:** BI, Power BI and Tableau developers, ETL developers, analytics, data specialists and data management, quantitative and MIS analysts, big data and Hadoop | Jacob: almost no data-and-analytics career changers; rejections were throwing away people the model can learn from | *not reviewed* |
| 7 | **More date formats:** MM/YY ("06/15", read as month/year; 2-digit years up to the dataset's year are 20xx), YYYY/MM, a year with a trailing slash, MM/DD/YYYY, and month names ("June 2015"); "Till Date" counts as present. A month above 12 stays unreadable | 3b: 37% of people had no readable job date; MM/YY alone was 83% of unreadable start dates, YYYY/MM 13% | *not reviewed* |
| 8 | **Manager-level first tech jobs count** (level is a label and a feature; prior experience can qualify someone). **New pool, changed roles within tech:** someone already in tech who moves into a different role bucket after 12+ months in the first, with that move as the switch (`from_bucket` records where they came from) | Jacob: why filter on level at all; career changers include role changes within tech, and data roles are mostly reached that way | *to fill in after the next spot-check* |

## 6. Freeze and save

*Once the rules pass (90%+ precision two rounds in a row):* save the candidates with their pre-switch profiles (jobs and education before the switch month only) to `data/profiles/switchers_v<RULES_VERSION>.pkl` for the ML step, and log the decision in `docs/decisions.md`.